In [1]:
from dotenv import load_dotenv
load_dotenv()

True

# 필수 요소
@tool  이라는 데코레이션


name 함수의 이름 ex) def call_api


description   함수의 설명


expected arguments 자료형을 더한 매개변수

In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o')
small_llm = ChatOpenAI(model='gpt-4o-mini')

In [24]:
from langchain_core.tools import tool

@tool  #tool 이라는 decorator함수 사용
def add (a: int, b: int) -> int:   #name 함수의 이름 , expected arguments 자료형을 더한 매개변수
    """숫자 a와 b를 더한 결과를 반환합니다."""   #description 함수의 설명
    return a + b

#일반 함수 tool이라는 데코레이션 사용하지 않음

def multiply(a: int, b: int) -> int:
    """숫자 a와 b를 곱한 결과를 반환합니다."""
    return a * b

In [6]:
multiply(4,6)

24

In [ ]:
#add(4,6) # add 는 더이상 일반 함수가 아닌 "Tool" 이라는 특별한 함수가 됨 따라서 이 부분은 에러발생

TypeError: 'StructuredTool' object is not callable

In [ ]:
add.invoke({'a':4, 'b':6}) #이와 같이 invoke 함수를 사용해서 호출해야함, 그리고 인자는 딕셔너리 형태로 전달해야함

10

In [25]:
@tool
def multiply(a: int, b: int) -> int:
    """숫자 a와 b를 곱한 결과를 반환합니다."""
    return a * b

In [26]:
llm_with_tools = small_llm.bind_tools([add, multiply]) # 이전 function calling 방식과 달리 더 편리한 tool 사용 방식


In [10]:
query = "4와 6을 더하면 얼마인가요?"

In [ ]:
small_llm.invoke(query) #이러면 AI가 계산후 답변해줌

AIMessage(content='4와 6을 더하면 10입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 18, 'total_tokens': 29, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_1590f93f9d', 'id': 'chatcmpl-D3dGmJoOemyehPzSgMSnNbsCAdvpI', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c0dd0-b37a-7e23-ae16-a21b4a05c708-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 11, 'total_tokens': 29, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [14]:
result = llm_with_tools.invoke(query) #이러면 답변에 additional_kwargs

In [ ]:
result.tool_calls   #이렇게 tool_calls에 리스트로 담긴다.

[{'name': 'add',
  'args': {'a': 4, 'b': 6},
  'id': 'call_YCJgjVwro8EDV17DWhHjOMzI',
  'type': 'tool_call'}]

In [18]:
from typing import Sequence
from langchain_core.messages import AnyMessage, HumanMessage

human_message = HumanMessage(query)
message_list: Sequence[AnyMessage] = [human_message]


In [19]:
ai_message = llm_with_tools.invoke(message_list)

In [20]:
ai_message.tool_calls

[{'name': 'add',
  'args': {'a': 4, 'b': 6},
  'id': 'call_W2q4KALTBE1UNXob73DFyNCk',
  'type': 'tool_call'}]

In [21]:
message_list.append(ai_message)

In [28]:
tool_message =multiply.invoke(ai_message.tool_calls[0])

In [29]:
message_list.append(tool_message)

In [30]:
llm_with_tools.invoke(message_list)

AIMessage(content='4와 6을 더하면 24입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 120, 'total_tokens': 132, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_1590f93f9d', 'id': 'chatcmpl-D3dSXJiZKM0x12rILllBtfXlrk6Pc', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c0ddb-d50d-7eb2-912f-8cfa18c55935-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 120, 'output_tokens': 12, 'total_tokens': 132, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})